<a href="https://colab.research.google.com/github/umang0015/Ai-DataScience/blob/main/Generative%20AI/28AugSession_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
POLICY_TEXT = """GenAI Cohort Program -- Policies & Logistics

Enrollment & Refunds
Refunds are available within 11 business days of enrollment. Refund requests must be submitted in writing to the program coordinator. Processing takes 5 to 7 business days once approved.

Attendance Policy
Students are expected to attend at least 12 of the 14 scheduled training days to remain eligible for a certificate of completion. Refund eligibility is void once a student has attended more than 2 live sessions, regardless of the 11-business-day window above. Attendance is tracked automatically through the Colab session logs.

Program Format
The program runs across 14 training days using a teach-practice-teach-practice format, split into 7 parts. Each topic includes a theory document, a hands-on notebook, and a set of practice tasks. Sessions run live, with recordings posted within 24 hours.

Capstone Project
The capstone is a post-program extension lasting 1 to 2 weeks. Topic selection for the capstone begins around day 7 to day 9 of the live training, once students have covered embeddings, RAG, and basic agent concepts.

Certification
A certificate of completion is issued after the capstone project is reviewed and approved by a mentor. Certificates are issued within 10 business days of capstone submission. Certificates cannot be reissued once a refund has been processed.

Support & Office Hours
Office hours are held twice a week, on Tuesdays and Thursdays, for one hour each session. Questions outside office hours can be posted in the cohort's shared forum, with a typical response time of one business day."""

print("Number of documents:", len(POLICY_TEXT) )

Number of documents: 1601


In [ ]:
!pip install -q sentence-transformers faiss-cpu pypdf fpdf2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.0/337.0 kB 20.7 MB/s eta 0:00:00


In [ ]:
# import libraries
import faiss
import numpy as np
# creating the pdf
from fpdf import FPDF
# reading the pdf
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer


In [ ]:
# now create a pdf
pdf = FPDF()
#
pdf.add_page()
pdf.set_font("Arial" , size=12)
pdf.multi_cell(0,10,POLICY_TEXT)
pdf.output("policy.pdf")
print("doc created")

doc created


/tmp/ipykernel_1694/1023913843.py:5: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font("Arial" , size=12)


In [ ]:
# Ingest the data
pdf_reader = PdfReader("policy.pdf")
text = ""
for page in pdf_reader.pages:
    text += page.extract_text()
    # reading page by page
    # print lenght of extracted text
print(len(text))
print("no of pages: " , len(pdf_reader.pages))


1594
no of pages:  2


In [ ]:
# define naive chunking function to chunk data based on no of characters
def naive_chunking(text , chunk_size = 100):
  chunk = [text[i:i+chunk_size]for i in range(0,len(text) , chunk_size)]
  return chunk
  # keyword arguments vs postional arguments *****

  # call the function and print few chunks
chunks = naive_chunking(text)
print(len(chunks))
print(chunks[0])
print(chunks[1])
print(chunks[2])

16
GenAI Cohort Program -- Policies & Logistics
Enrollment & Refunds
Refunds are available within 11 bu
siness days of enrollment. Refund requests must be submitted in
writing to the program coordinator. 
Processing takes 5 to 7 business days once approved.
Attendance Policy
Students are expected to atte


##**Sentence aware chunking**
- overlap chuking
- look for specifies like paragraph end , page end , topic start and then apply chunking

In [ ]:
# pass to encoder (embedding model)
encoder = SentenceTransformer("BAAI/bge-small-en-v1.5")
chunks_vectors = encoder.encode(chunks)
print(chunks_vectors.shape)
# add to faiss DB
index = faiss.IndexFlatL2(chunks_vectors.shape[1])
index.add(chunks_vectors)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(16, 384)


In [ ]:
# query
Query= "if i attend classes for 10 days will i get certificate or not"

In [ ]:
# Query vector creation and retrieval
Query_vector = encoder.encode([Query])
# pick the top 3 chunks
scores , indices = index.search(Query_vector,3)
for i in indices[0]:
  print(chunks[i])

proved by a
mentor. Certificates are issued within 10 business days of capstone submission. Certific
nd at least 12 of the 14 scheduled training days to remain eligible for a
certificate of completion.
Processing takes 5 to 7 business days once approved.
Attendance Policy
Students are expected to atte
